# Fair carbon-aware MARLISA

This notebook adds occupied thermal comfort and cross-building fairness to the carbon-aware MARLISA reward, then compares all five approaches developed so far:

1. `BasicRBC`;
2. independent SAC;
3. MARLISA;
4. carbon-aware MARLISA; and
5. fair carbon-aware MARLISA.

The main question is whether the fair controller can retain useful carbon reduction while substantially improving the experience of the worst-served building.


## Reward design

For each occupied building, define the temperature deviation outside a comfort band $b$ as:

$$d_i = \max(|T_i-T_i^{set}|-b,0)$$

and the occupied comfort loss as:

$$L_i = \mathbb{1}_{occupied}\mathbb{1}_{d_i>0}(1+d_i).$$

The shared fairness loss is:

$$F = \max_i L_i + (\max_i L_i-\min_i L_i).$$

The first term protects the currently worst building. The second penalizes the gap between buildings. Each decentralized agent receives:

$$r_i=-w_c C_i-w_hL_i-w_fF,$$

where $C_i=c_t\max(E_i,0)$ is operational carbon emissions. The development weights are $w_c=1.0$, $w_h=0.25$, $w_f=0.25$, with a 1°C band. They are explicit experimental choices rather than tuned final values.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

VENV_SITE_PACKAGES = PROJECT_ROOT / '.venv' / 'Lib' / 'site-packages'
if VENV_SITE_PACKAGES.is_dir() and str(VENV_SITE_PACKAGES) not in sys.path:
    sys.path.append(str(VENV_SITE_PACKAGES))

from src.rewards import FairCarbonAwareReward
from src.run_carbon_aware_marlisa import run_carbon_aware_marlisa
from src.run_fair_carbon_aware_marlisa import run_fair_carbon_aware_marlisa
from src.run_independent_sac import run_independent_sac
from src.run_marlisa import run_marlisa
from src.run_rbc import run_basic_rbc

SCHEMA = PROJECT_ROOT / 'notebooks' / 'data' / 'schema.json'
print(f'Project root: {PROJECT_ROOT}')
print(f'Schema exists: {SCHEMA.is_file()}')


## Check the reward on a simple example

Building 1 is occupied and 1°C beyond the permitted band. Building 2 is comfortable. The shared fairness loss affects both rewards, while only Building 1 receives the local comfort loss.


In [ ]:
reward_example = FairCarbonAwareReward({'central_agent': False})
example = [
    {
        'net_electricity_consumption': 2.0,
        'carbon_intensity': 0.5,
        'indoor_dry_bulb_temperature': 26.0,
        'indoor_dry_bulb_temperature_cooling_set_point': 24.0,
        'occupant_count': 2.0,
    },
    {
        'net_electricity_consumption': 1.0,
        'carbon_intensity': 0.5,
        'indoor_dry_bulb_temperature': 24.0,
        'indoor_dry_bulb_temperature_cooling_set_point': 24.0,
        'occupant_count': 1.0,
    },
]
pd.DataFrame(example).assign(reward=reward_example.calculate(example))


## Experimental configuration

All learned controllers use the same 720-step data, seed 42, network sizes, optimizer settings, and three-episode schedule. Three episodes contain one initial exploration episode, one learning episode, and one deterministic evaluation episode.


In [ ]:
SEED = 42
TRAINING_EPISODES = 3
RETRAIN_EXISTING = False
RUN_MISSING_BASELINES = True

SAC_PARAMETERS = {
    'hidden_dimension': (256, 256),
    'discount': 0.99,
    'tau': 0.005,
    'alpha': 0.2,
    'learning_rate': 3e-4,
    'batch_size': 256,
    'replay_buffer_capacity': 100_000,
    'action_scaling_coefficient': 0.5,
    'reward_scaling': 5.0,
    'update_per_time_step': 1,
}
MARLISA_PARAMETERS = {
    'regression_buffer_capacity': 30_000,
    'start_regression_time_step': 2,
    'regression_frequency': 2_500,
    'information_sharing': True,
    'pca_compression': 1.0,
    'iterations': 2,
}
FAIR_REWARD_PARAMETERS = {
    'carbon_weight': 1.0,
    'comfort_weight': 0.25,
    'fairness_weight': 0.25,
    'comfort_band': 1.0,
}
OUTPUTS = {
    'BasicRBC': PROJECT_ROOT / 'outputs' / 'rbc',
    'Independent SAC': PROJECT_ROOT / 'outputs' / 'independent_sac',
    'MARLISA': PROJECT_ROOT / 'outputs' / 'marlisa',
    'Carbon-aware MARLISA': PROJECT_ROOT / 'outputs' / 'carbon_aware_marlisa',
    'Fair carbon-aware MARLISA': PROJECT_ROOT / 'outputs' / 'fair_carbon_aware_marlisa',
}


## Create any missing results

Existing canonical results are reused. Set `RETRAIN_EXISTING=True` only when intentionally replacing every result with a matched rerun. Training the four learning controllers can take several minutes.


In [ ]:
def result_exists(controller):
    return (OUTPUTS[controller] / 'kpis.csv').is_file()

if RETRAIN_EXISTING or not result_exists('BasicRBC'):
    run_basic_rbc(SCHEMA, OUTPUTS['BasicRBC'], SEED)

if RUN_MISSING_BASELINES and (RETRAIN_EXISTING or not result_exists('Independent SAC')):
    print('Training independent SAC...')
    run_independent_sac(
        SCHEMA, OUTPUTS['Independent SAC'], TRAINING_EPISODES, SEED,
        reward='independent_sac', **SAC_PARAMETERS,
    )

if RUN_MISSING_BASELINES and (RETRAIN_EXISTING or not result_exists('MARLISA')):
    print('Training MARLISA...')
    run_marlisa(
        SCHEMA, OUTPUTS['MARLISA'], TRAINING_EPISODES, SEED,
        reward='independent_sac', **SAC_PARAMETERS, **MARLISA_PARAMETERS,
    )

if RUN_MISSING_BASELINES and (RETRAIN_EXISTING or not result_exists('Carbon-aware MARLISA')):
    print('Training carbon-aware MARLISA...')
    run_carbon_aware_marlisa(
        schema=SCHEMA, output_directory=OUTPUTS['Carbon-aware MARLISA'],
        episodes=TRAINING_EPISODES, random_seed=SEED,
        **SAC_PARAMETERS, **MARLISA_PARAMETERS,
    )

if RETRAIN_EXISTING or not result_exists('Fair carbon-aware MARLISA'):
    print('Training fair carbon-aware MARLISA...')
    fair_result = run_fair_carbon_aware_marlisa(
        schema=SCHEMA, output_directory=OUTPUTS['Fair carbon-aware MARLISA'],
        episodes=TRAINING_EPISODES, random_seed=SEED,
        **SAC_PARAMETERS, **MARLISA_PARAMETERS, **FAIR_REWARD_PARAMETERS,
    )
else:
    fair_result = None
    print('Reusing existing fair carbon-aware MARLISA results.')

pd.Series({name: result_exists(name) for name in OUTPUTS}, name='results_available')


## Confirm comparability

The learned runs should match on seed, time steps, episodes, batch size, learning rate, and updates per step. Their reward or controller architecture is the intended difference. RBC actions are rule based and do not depend on the reward.


In [ ]:
context_rows = []
for controller, directory in OUTPUTS.items():
    metadata = json.loads((directory / 'metadata.json').read_text(encoding='utf-8'))
    sac = metadata.get('sac_hyperparameters', metadata.get('hyperparameters', {}))
    context_rows.append({
        'controller': controller,
        'seed': metadata.get('random_seed'),
        'time_steps': metadata.get('episode_time_steps', metadata.get('time_steps')),
        'episodes': metadata.get('episodes', 1),
        'reward': metadata.get('reward_choice', 'not used for RBC actions'),
        'batch_size': sac.get('batch_size'),
        'learning_rate': sac.get('learning_rate'),
        'updates_per_step': sac.get('update_per_time_step'),
    })
experiment_context = pd.DataFrame(context_rows).set_index('controller')
display(experiment_context)


## Current saved development results

These results use seed 42, 720 time steps, and three episodes. Lower is better.

| District metric | RBC | SAC | MARLISA | Carbon-aware MARLISA | Fair carbon-aware MARLISA |
| --- | ---: | ---: | ---: | ---: | ---: |
| Carbon emissions ratio | 1.890 | **0.442** | 0.448 | 0.449 | 0.873 |
| Cost ratio | 1.843 | **0.415** | 0.421 | 0.422 | 0.856 |
| Electricity ratio | 1.900 | **0.448** | 0.454 | 0.456 | 0.877 |
| Daily peak ratio | 1.329 | **0.713** | 0.716 | 0.716 | 0.910 |
| Ramping ratio | 0.924 | 0.888 | 0.887 | **0.887** | 1.098 |

| Comfort/fairness metric | RBC | SAC | MARLISA | Carbon-aware MARLISA | Fair carbon-aware MARLISA |
| --- | ---: | ---: | ---: | ---: | ---: |
| Average discomfort | 0.947 | 0.980 | 0.974 | 0.979 | **0.201** |
| Worst-building discomfort | 0.975 | 0.982 | 0.982 | 0.982 | **0.336** |
| Best-building discomfort | 0.908 | 0.978 | 0.961 | 0.976 | **0.133** |
| Worst-best discomfort gap | 0.066 | **0.004** | 0.021 | 0.006 | 0.203 |


## Load KPI data


In [ ]:
kpis_by_controller = {
    controller: pd.read_csv(directory / 'kpis.csv')
    for controller, directory in OUTPUTS.items()
}
print(f'Loaded {len(kpis_by_controller)} controller results.')


## District energy and carbon comparison

The ratios are relative to CityLearn's no-control reference. Values below 1 improve on that reference.


In [ ]:
DISTRICT_METRICS = [
    'carbon_emissions_total',
    'cost_total',
    'electricity_consumption_total',
    'daily_peak_average',
    'ramping_average',
    'one_minus_thermal_resilience_proportion',
]
district_comparison = pd.concat({
    controller: kpis.query("level == 'district'").set_index('cost_function')['value']
    for controller, kpis in kpis_by_controller.items()
}, axis=1).loc[DISTRICT_METRICS]
display(district_comparison.round(3))

ax = district_comparison.drop(index='one_minus_thermal_resilience_proportion').plot.bar(figsize=(13, 5))
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='No-control reference')
ax.set_ylabel('KPI ratio (lower is better)')
ax.set_title('District energy and carbon KPIs')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## Improvement relative to RBC

Positive percentages mean that the controller reduced the KPI relative to RBC. A negative ramping result means the controller increased ramping.


In [ ]:
rbc_values = district_comparison['BasicRBC']
improvement_vs_rbc = district_comparison.drop(columns='BasicRBC').apply(
    lambda values: 100.0 * (rbc_values - values) / rbc_values
)
display(improvement_vs_rbc.round(1).style.format('{:.1f}%'))


## Building comfort and fairness

Average discomfort measures overall service. Worst-building discomfort protects the least-served building. The worst-best gap measures equality, but it must be read together with the absolute discomfort values: uniformly poor service can produce a very small gap.


In [ ]:
fairness_rows = []
building_discomfort = {}
for controller, kpis in kpis_by_controller.items():
    discomfort = kpis.loc[
        (kpis['level'] == 'building')
        & (kpis['cost_function'] == 'discomfort_proportion'),
        ['name', 'value'],
    ].dropna()
    building_discomfort[controller] = discomfort.set_index('name')['value']
    worst = discomfort.loc[discomfort['value'].idxmax()]
    best = discomfort.loc[discomfort['value'].idxmin()]
    fairness_rows.append({
        'controller': controller,
        'average_discomfort': discomfort['value'].mean(),
        'worst_discomfort': worst['value'],
        'best_discomfort': best['value'],
        'discomfort_gap': worst['value'] - best['value'],
        'worst_building': worst['name'],
    })
fairness_comparison = pd.DataFrame(fairness_rows).set_index('controller')
building_discomfort = pd.concat(building_discomfort, axis=1)
display(fairness_comparison.round(3))
display(building_discomfort.round(3))

ax = building_discomfort.plot.bar(figsize=(12, 5))
ax.set_ylabel('Discomfort proportion (lower is better)')
ax.set_title('Building-level occupied discomfort')
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


## Carbon-comfort trade-off and Pareto comparison

A point is dominated when another controller has both lower carbon and lower worst-building discomfort. The useful frontier contains controllers representing different compromises rather than one universally best policy.


In [ ]:
tradeoff = pd.DataFrame({
    'district_carbon': district_comparison.loc['carbon_emissions_total'],
    'worst_discomfort': fairness_comparison['worst_discomfort'],
    'average_discomfort': fairness_comparison['average_discomfort'],
    'discomfort_gap': fairness_comparison['discomfort_gap'],
}).dropna()

def is_dominated(controller, table):
    candidate = table.loc[controller]
    others = table.drop(index=controller)
    no_worse = (others['district_carbon'] <= candidate['district_carbon']) & (others['worst_discomfort'] <= candidate['worst_discomfort'])
    strictly_better = (others['district_carbon'] < candidate['district_carbon']) | (others['worst_discomfort'] < candidate['worst_discomfort'])
    return bool((no_worse & strictly_better).any())

tradeoff['pareto_efficient'] = [not is_dominated(name, tradeoff) for name in tradeoff.index]
display(tradeoff.round(3))

fig, ax = plt.subplots(figsize=(9, 5))
for controller, row in tradeoff.iterrows():
    marker = '*' if row['pareto_efficient'] else 'o'
    size = 180 if row['pareto_efficient'] else 70
    ax.scatter(row['district_carbon'], row['worst_discomfort'], marker=marker, s=size)
    ax.annotate(controller, (row['district_carbon'], row['worst_discomfort']), xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('District carbon-emissions ratio (lower is better)')
ax.set_ylabel('Worst-building discomfort proportion (lower is better)')
ax.set_title('Carbon versus worst-building comfort (* = Pareto efficient)')
ax.grid(alpha=0.25)
plt.show()


## Discussion: carbon and energy performance

Fair carbon-aware MARLISA achieved a district carbon ratio of **0.873**, a **53.8% reduction relative to RBC** and a 12.7% reduction relative to CityLearn's no-control reference. It therefore retained meaningful carbon savings after comfort and fairness were introduced. Its electricity and cost ratios followed the same pattern at 0.877 and 0.856.

The aggressive electricity-based controllers achieved much lower carbon ratios of 0.442–0.449. Fair carbon-aware MARLISA used more electricity to maintain indoor temperatures, so its carbon ratio was 94.3% higher than carbon-aware MARLISA. This is an expected objective trade-off rather than evidence that the carbon term is absent.

Daily peak demand remained below the no-control reference at 0.910. Ramping increased to 1.098, however, indicating more abrupt changes in grid consumption. A final objective may need a ramping or peak term if grid flexibility is part of the project scope.


## Discussion: comfort and fairness

The fair reward produced the clearest improvement in occupied comfort. Average building discomfort fell from **0.979 to 0.201** relative to carbon-aware MARLISA, a 79.5% reduction. Worst-building discomfort fell from **0.982 to 0.336**, a 65.7% reduction. Building 2 remained the worst-served building, while Buildings 1 and 3 both reached approximately 0.133 discomfort.

The worst-best gap increased to 0.203. This initially looks less fair than the 0.006 gap under carbon-aware MARLISA, but the small earlier gap resulted from all three buildings being uncomfortable almost all the time. Equality at a uniformly poor outcome is not desirable. The fair controller greatly improved the absolute outcome of every building and especially the worst building, while leaving a remaining allocation gap that should be addressed in later tuning.

This is why the project should report at least three quantities together: average discomfort, worst-building discomfort, and the gap. The gap alone would rank the uniformly poor SAC policy as highly fair.


## Discussion: answer to the research question

For this development run, coordinated reinforcement learning with an explicit fairness-aware comfort term reduced district carbon relative to RBC while substantially improving both average and worst-building comfort. The result demonstrates that the reward can move the policy toward a more useful carbon-comfort compromise.

It does not yet establish that fair carbon-aware MARLISA is generally superior. Independent SAC remained best for carbon under this short budget, and fair carbon-aware MARLISA produced a larger between-building discomfort gap and worse ramping. The result supports the proposed approach as a candidate trade-off policy that requires longer training, weight selection, and held-out evaluation.


## Threats to validity

- **Training length:** each learned controller has only one effective learning episode after exploration. Neural policies are unlikely to have converged.
- **Single seed:** differences may reflect stochastic initialization and replay sampling.
- **In-sample evaluation:** the deterministic episode repeats the same 720-step period used during training.
- **Reward weights:** 1.0/0.25/0.25 were chosen as interpretable starting values and were not selected through a validation set.
- **Instantaneous fairness:** the reward uses current temperature loss, while the reported fairness metrics aggregate discomfort across the episode.
- **Dataset scope:** the current three-building pilot period may not represent other seasons, climates, or building types.
- **Operational definition:** fairness refers to equality of simulated building comfort outcomes and does not represent fairness across demographic groups.


## Recommended next experiments

1. Define separate training, validation, and held-out test periods.
2. Train each learned controller for the same larger number of transitions over at least three to five seeds.
3. Sweep `comfort_weight` and `fairness_weight` on validation data and plot a Pareto frontier for carbon versus worst-building discomfort.
4. Investigate Building 2, which remains the worst-served building, using its set points, occupancy, cooling demand, device capacity, and action history.
5. Add ramping or peak penalties only if those outcomes are part of the final research question.
6. Report mean, standard deviation or confidence intervals, and per-building results on held-out data.
